In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:
df = pd.read_csv("korba_defects.csv")  # ← same filename as before

print("Data loaded!")
print("Shape:", df.shape)

Data loaded!
Shape: (8943, 21)


### STEP1 - DROPPING USELESS COLUMNS

In [6]:
# Printing exact column names with their length
# This reveals any hidden spaces or special characters

for col in df.columns:
    print(f"'{col}'  → length: {len(col)}")

'Notif.date'  → length: 10
'Planner group'  → length: 13
'Functional Loc.'  → length: 15
'equipment'  → length: 9
'Created On'  → length: 10
'Notif. Time'  → length: 11
'Completion time'  → length: 15
'Completn date'  → length: 13
'Planning plant'  → length: 14
'Priority'  → length: 8
'Notifictn type'  → length: 14
'Plnt WorkCenter'  → length: 15


In [8]:
print("Total columns remaining:", df.shape[1])

Total columns remaining: 12


In [11]:
# Notif.date → integer format YYYYMMDD
df['Notif.date'] = pd.to_datetime(df['Notif.date'], format='%Y%m%d')

# Created On → DD/MM/YYYY (slash not dash)
df['Created On'] = pd.to_datetime(df['Created On'], format='%d/%m/%Y', errors='coerce')

# Completn date → let pandas figure out the format automatically
df['Completn date'] = pd.to_datetime(df['Completn date'], format='mixed', dayfirst=True, errors='coerce')

print("Date columns parsed successfully!")

Date columns parsed successfully!


### STEP 2 - CREATING NEW COLUMNS SUCH THAT WHICH WILL BW USED FOR ML TRAINING

In [14]:
### extracting year, month and day of the week
df['year'] = df['Notif.date'].dt.year # PARAMETER 1
df['month'] = df['Notif.date'].dt.month
df['day_of_week'] = df['Notif.date'].dt.dayofweek  # 0=Monday, 6=Sunday

### extracting the exact hour
df['hour'] = pd.to_datetime(df['Notif. Time'], format='%H:%M:%S', errors='coerce').dt.hour

## creating a shift column to understand the between whcih time period the most time defects occur 
def get_shift(hour):
    if pd.isnull(hour):
        return "unknown"
    elif 6 <= hour <= 13:
        return 'Morning'
    elif 14 <= hour <= 21:
        return 'Afternoon'
    else:
        return 'Night'

df['shift'] = df['hour'].apply(get_shift)
print("New features extracted successfully!")
print(df[['Notif.date','year','month','hour','day_of_week','shift']].head(5))

New features extracted successfully!
  Notif.date  year  month  hour  day_of_week      shift
0 2014-01-01  2014      1    12            2    Morning
1 2014-01-01  2014      1    12            2    Morning
2 2014-01-02  2014      1    10            3    Morning
3 2014-01-02  2014      1    17            3  Afternoon
4 2014-01-05  2014      1     4            6      Night


### STEP 3 - CREATING MORE PARAMETERS AND UNDERSTANDING THE DATA

In [16]:

df['Completn date'] = pd.to_datetime(df['Completn date'], errors='coerce')
df['resolution_days'] = (df['Completn date'] - df['Notif.date']).dt.days
df.loc[df['resolution_days'] < 0, 'resolution_days'] = None

print("Resolution time calculated!")
print(df[['Notif.date', 'Completn date', 'resolution_days']].head(5))


Resolution time calculated!
  Notif.date Completn date  resolution_days
0 2014-01-01    2014-06-01            151.0
1 2014-01-01    2014-06-01            151.0
2 2014-01-02    2014-03-01             58.0
3 2014-01-02    2014-11-03            305.0
4 2014-01-05    2014-09-01            239.0


### STEP 4- FILLING UP THE MISSING  CELL VALUES

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8943 entries, 0 to 8942
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Notif.date       8943 non-null   datetime64[ns]
 1   Planner group    8933 non-null   object        
 2   Functional Loc.  8943 non-null   object        
 3   equipment        8943 non-null   object        
 4   Created On       3595 non-null   datetime64[ns]
 5   Notif. Time      8943 non-null   object        
 6   Completion time  8721 non-null   object        
 7   Completn date    8633 non-null   datetime64[ns]
 8   Planning plant   8651 non-null   float64       
 9   Priority         8591 non-null   object        
 10  Notifictn type   8651 non-null   object        
 11  Plnt WorkCenter  8651 non-null   float64       
 12  year             8943 non-null   int32         
 13  month            8943 non-null   int32         
 14  day_of_week      8943 non-null   int32  

In [19]:
df.columns

Index(['Notif.date', 'Planner group', 'Functional Loc.', 'equipment',
       'Created On', 'Notif. Time', 'Completion time', 'Completn date',
       'Planning plant', 'Priority', 'Notifictn type', 'Plnt WorkCenter',
       'year', 'month', 'day_of_week', 'hour', 'shift', 'resolution_days'],
      dtype='object')

In [22]:
# Dropping Planner group — not required for ML
df = df.drop(columns=['Planner group','Planning plant','Plnt WorkCenter',], errors='ignore')

#### FILLED THE EMPTY CELLS

In [24]:
# Priority — fill with most common value (Priority A)
df['Priority'] = df['Priority'].fillna(df['Priority'].mode()[0])

# Notifictn type — fill with most common value
df['Notifictn type'] = df['Notifictn type'].fillna(df['Notifictn type'].mode()[0])

# resolution_days — fill with median
df['resolution_days'] = df['resolution_days'].fillna(df['resolution_days'].median())

#### CHECKING THE COUNT OF NULL VALUES

In [26]:
null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0]
if len(null_counts) == 0:
    print("  Zero nulls remaining — perfect!")
else:
    print(null_counts)


Created On         5348
Completion time     222
Completn date       310
dtype: int64


In [27]:
# Drop Created On column — too many nulls + Notif.date already covers it
df = df.drop(columns=['Created On'], errors='ignore')

# Drop rows where Completion time or Completn date is null
df = df.dropna(subset=['Completion time', 'Completn date'])

print("Done!")
print("\nShape after cleaning:", df.shape)


null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0]
if len(null_counts) == 0:
    print("  Zero nulls remaining — perfect!")
else:
    print(null_counts)

print("\nFinal columns:")
for i, col in enumerate(df.columns):
    print(i, "→", col)

Done!

Shape after cleaning: (8633, 14)

Null counts:
  Zero nulls remaining — perfect!

Final columns:
0 → Notif.date
1 → Functional Loc.
2 → equipment
3 → Notif. Time
4 → Completion time
5 → Completn date
6 → Priority
7 → Notifictn type
8 → year
9 → month
10 → day_of_week
11 → hour
12 → shift
13 → resolution_days


### STEP 5 -# ML models only understand numbers — not text
### So we convert text columns to numbers using LabelEncoder

In [ ]:

from sklearn.preprocessing import LabelEncoder
import pickle

# Columns that need encoding (text → number)
cols_to_encode = ['equipment', 'Functional Loc.', 'Priority', 'Notifictn type', 'shift']

# Dictionary to store all encoders
# We save them so we can decode predictions later in the app
label_encoders = {}

for col in cols_to_encode:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le
    print(f"{col:20} → encoded ({len(le.classes_)} unique values)")

print("\nEncoding complete!")
print("\nSample after encoding:")
print(df[['equipment', 'Functional Loc.', 'Priority', 'Notifictn type', 'shift']].head(3)) 

In [30]:
# Logic: for each equipment, sort by date
#        then calculate how many days until the NEXT defect

# Sort by equipment and date
df = df.sort_values(['equipment', 'Notif.date'])

# For each equipment group, shift the date up by 1
# This gives us the date of the NEXT defect for that equipment
df['next_defect_date'] = df.groupby('equipment')['Notif.date'].shift(-1)

# Calculate days between current defect and next defect
df['days_to_next_defect'] = (df['next_defect_date'] - df['Notif.date']).dt.days

# The last defect record of each equipment will have no next defect
# So we drop those rows
df = df.dropna(subset=['days_to_next_defect'])

# Convert to integer
df['days_to_next_defect'] = df['days_to_next_defect'].astype(int)

# Remove unrealistic values
# If days_to_next_defect is 0 or negative → data issue → remove
df = df[df['days_to_next_defect'] > 0]

print("Target column created!")
print("\ndays_to_next_defect summary:")
print(df['days_to_next_defect'].describe())
print("\nSample:")
print(df[['equipment', 'Notif.date', 'next_defect_date', 'days_to_next_defect']].head(5))

Target column created!

days_to_next_defect summary:
count    7693.000000
mean      145.098791
std       252.261612
min         1.000000
25%        13.000000
50%        46.000000
75%       160.000000
max      3425.000000
Name: days_to_next_defect, dtype: float64

Sample:
      equipment Notif.date next_defect_date  days_to_next_defect
3258          0 2019-06-13       2021-05-27                  714
6347          1 2023-01-21       2023-03-14                   52
6503          2 2023-03-21       2023-04-11                   21
6547          2 2023-04-11       2023-09-15                  157
7035          2 2023-09-15       2023-09-28                   13


###  Remove Outliers from Target - Because we can here see that max gap is almost 3425 days almost 9 years which rarely happens we need to cap it otherwise a trouble for ML

In [31]:
df = df[df['days_to_next_defect'] <= 365]

print("Outliers removed!")
print("\nUpdated days_to_next_defect summary:")
print(df['days_to_next_defect'].describe())
print("\nRemaining rows:", df.shape[0])


Outliers removed!

Updated days_to_next_defect summary:
count    6846.000000
mean       73.276658
std        86.349796
min         1.000000
25%        11.000000
50%        36.000000
75%       106.000000
max       365.000000
Name: days_to_next_defect, dtype: float64

Remaining rows: 6846


In [32]:
# Saving the clean data to a CSV file
# This will be used in step3_model.ipynb for training

# Columns to save for ML training
ml_columns = [
    'equipment',
    'Functional Loc.',
    'Priority',
    'Notifictn type',
    'month',
    'day_of_week',
    'hour',
    'shift',
    'year',
    'days_to_next_defect'  # target column
]

df_ml = df[ml_columns]
df_ml.to_csv('clean_ntpc_data.csv', index=False)

print("Clean data saved as clean_ntpc_data.csv!")
print("Shape:", df_ml.shape)
print("\nSample:")
print(df_ml.head(3))

Clean data saved as clean_ntpc_data.csv!
Shape: (6846, 10)

Sample:
      equipment  Functional Loc.  Priority  Notifictn type  month  \
6347          1              446         1               0      1   
6503          2              447         1               1      3   
6547          2              447         1               1      4   

      day_of_week  hour  shift  year  days_to_next_defect  
6347            5    11      1  2023                   52  
6503            1    10      1  2023                   21  
6547            1    11      1  2023                  157  


In [34]:
# Saving label encoders so we can use them in the app later
# Without this we won't know that equipment 1 = which equipment name

import pickle

pickle.dump(label_encoders, open('label_encoders.pkl', 'wb'))

print("Label encoders saved as label_encoders.pkl!")
print("\nEncoders saved for these columns:")
for col in label_encoders:
    print(f"  {col:20} → {len(label_encoders[col].classes_)} unique values")

print("\nStep 2 — Cleaning Complete! ")
print("Files created:")
print("  → clean_ntpc_data.csv")
print("  → label_encoders.pkl")

Label encoders saved as label_encoders.pkl!

Encoders saved for these columns:
  equipment            → 447 unique values
  Functional Loc.      → 450 unique values
  Priority             → 5 unique values
  Notifictn type       → 6 unique values
  shift                → 3 unique values

Step 2 — Cleaning Complete! 
Files created:
  → clean_ntpc_data.csv
  → label_encoders.pkl
